# Sanity checks: metrics 1 and 2

Hand checks for `src/metrics.py`, plus both versions of each open decision so they can be compared. Definitions are in `METHOD.md`.

In [1]:
import sys; sys.path.insert(0, '..')
import pandas as pd
from src.metrics import load, retention_rows, retention, corner_deficit_rows, corner_deficit
pd.set_option('display.precision', 2)
results = load('results', {'R', 'S'})
corners = load('corners', {'Q', 'SQ'})
print(results.groupby(['SessionType'])['Round'].nunique().to_dict(), 'rounds by session type')

{'GP': 15, 'Sprint': 5} rounds by session type


## Metric 1: position retention

### Hand check: Azerbaijan GP (Round 15)
Expected from the official result: Antonelli 16th → 5th (+11), Verstappen 8th → 2nd (+6), Piastri 3rd → 13th (−10). Retired cars (e.g. Norris, both Aston Martins) should be missing; Bottas was "Retired" but classified 16th, so he should be included.

In [2]:
rows = retention_rows(results)
rows[(rows['Round'] == 15) & (rows['Session'] == 'R')].sort_values('Finish')

,Round,EventName,Session,SessionType,TeamName,Abbreviation,GridPosition,Finish,Gained,Lost,Expected,VsExpected
418,15,Azerbaijan Grand Prix,R,GP,Mercedes,RUS,1.0,1.0,0.0,0.0,-1.18,1.18
419,15,Azerbaijan Grand Prix,R,GP,Red Bull Racing,VER,8.0,2.0,6.0,0.0,-0.19,6.19
420,15,Azerbaijan Grand Prix,R,GP,Red Bull Racing,HAD,4.0,3.0,1.0,0.0,-0.76,1.76
421,15,Azerbaijan Grand Prix,R,GP,Ferrari,LEC,2.0,4.0,-2.0,-2.0,-1.18,-0.82
422,15,Azerbaijan Grand Prix,R,GP,Mercedes,ANT,16.0,5.0,11.0,0.0,2.31,8.69
423,15,Azerbaijan Grand Prix,R,GP,Ferrari,HAM,6.0,6.0,0.0,0.0,0.75,-0.75
424,15,Azerbaijan Grand Prix,R,GP,Racing Bulls,LIN,15.0,7.0,8.0,0.0,2.31,5.69
425,15,Azerbaijan Grand Prix,R,GP,Haas F1 Team,OCO,13.0,8.0,5.0,0.0,1.97,3.03
426,15,Azerbaijan Grand Prix,R,GP,Haas F1 Team,BEA,10.0,9.0,1.0,0.0,0.17,0.83
427,15,Azerbaijan Grand Prix,R,GP,Williams,SAI,14.0,10.0,4.0,0.0,1.97,2.03


### Hand check: one team computed by hand
Mercedes' net score should equal the plain average of its rows.

In [3]:
merc = rows[rows['TeamName'] == 'Mercedes']
print('by hand:', round(merc['Gained'].mean(), 3), '| function:', round(retention(results).loc['Mercedes', 'score'], 3), '| n =', len(merc))

by hand: 0.0 | function: 0.0 | n = 38


### Decision: how to count places
- **net**: average places gained (gains and losses both count).
- **lost**: average places lost (gains count as 0).
- **adjusted**: places gained compared with the typical car starting from the same grid slot (slots pooled in pairs: P1–2, P3–4, ...).

Higher is better in all three. Compare each with the team's average starting position: net and lost mostly track the grid (a car starting 20th can only gain, and gets promoted when cars ahead retire). Adjusted removes that built-in effect.

In [4]:
comp = pd.DataFrame({m: retention(results, m)['score'] for m in ['net', 'lost', 'adjusted']})
comp['avg_grid'] = rows.groupby('TeamName')['GridPosition'].mean()
comp = comp.sort_values('adjusted', ascending=False)
print('Spearman correlation with average grid position:', comp.corr(method='spearman')['avg_grid'].round(2).drop('avg_grid').to_dict())
comp

Spearman correlation with average grid position: {'net': 0.94, 'lost': 0.9, 'adjusted': -0.85}


,net,lost,adjusted,avg_grid
TeamName,,,,
Racing Bulls,1.72,-0.53,0.90,11.36
Mercedes,0.00,-1.21,0.53,3.42
McLaren,0.18,-0.91,0.42,4.91
Red Bull Racing,0.77,-1.26,0.35,7.77
Ferrari,0.05,-0.73,0.31,4.49
Alpine,1.22,-0.54,0.07,11.78
Haas F1 Team,1.61,-0.42,-0.25,14.42
Audi,0.81,-0.61,-0.30,12.35
Aston Martin,3.23,0.00,-0.56,19.64


### Expected places gained by starting slot
The baseline for **adjusted**: the average places gained by all drivers starting in each pair of grid slots, across every race and Sprint.

In [5]:
rows.groupby((rows['GridPosition'] - 1) // 2 * 2 + 1)['Expected'].first().rename('expected gain').rename_axis('grid slots from').round(1)

grid slots from
1.0    -1.2
3.0    -0.8
5.0     0.8
7.0    -0.2
9.0     0.2
11.0    0.5
13.0    2.0
15.0    2.3
17.0    2.1
19.0    3.8
21.0    5.1
Name: expected gain, dtype: float64

### Sprints vs Grand Prix (net)
Do teams behave differently in Sprints? (Sprints have about 22 rows per round vs about 22 per GP, but only 5 rounds.)

In [6]:
rows.pivot_table(index='TeamName', columns='SessionType', values='Gained', aggfunc='mean').sort_values('GP', ascending=False)

SessionType,GP,Sprint
TeamName,,
Aston Martin,4.15,1.89
Cadillac,2.89,0.44
Red Bull Racing,2.00,-2.30
Williams,1.91,1.80
Racing Bulls,1.68,1.88
Haas F1 Team,1.58,1.70
Alpine,1.48,0.50
Audi,1.24,-1.00
McLaren,0.22,0.10


## Metric 2: slow-corner speed

### Data quality: frozen speed readings
The 2026 speed channel sometimes stalls for up to ~1 s, holding one value (e.g. 305 km/h straight through a braking zone). Corner readings with 3+ identical consecutive speed samples are dropped, as are readings more than 30 km/h from the other teams at that corner. Corners are located by track-map position (X/Y), because FastF1's distance-along-lap drifts by up to ~3%.

If the dropped share is similar across teams, the cleaning doesn't favour anyone.

In [7]:
from src.metrics import clean_corner_speeds
cl = clean_corner_speeds(corners, 'Q1')
print(f"dropped overall: {(~cl['Valid']).mean():.1%}")
(~cl['Valid']).groupby(cl['Team']).mean().mul(100).round(1).sort_values().rename('dropped %')

dropped overall: 9.8%


Team
Cadillac            6.9
Haas F1 Team        7.6
Ferrari             7.9
Racing Bulls        8.3
Alpine              8.6
Williams            8.9
McLaren             9.6
Audi               11.2
Aston Martin       12.2
Mercedes           13.2
Red Bull Racing    13.2
Name: dropped %, dtype: float64

### Hand check: Azerbaijan qualifying, whole-session fastest lap
The Oct 3 test (before the cleaning) gave Mercedes 2.4 km/h behind the best team in each slow corner → Cadillac 8.5. The exact values change with the cleaning, but the order should be broadly similar: Mercedes near the front, Cadillac, Williams and Aston Martin near the back.

In [8]:
dr_all = corner_deficit_rows(corners, 'all')
dr_all[(dr_all['Round'] == 15)].sort_values('Deficit')[['Team', 'Deficit', 'SlowCorners', 'ValidCorners']]

,Team,Deficit,SlowCorners,ValidCorners
172,Mercedes,2.25,13,13
169,Ferrari,2.65,13,13
171,McLaren,3.69,13,13
170,Haas F1 Team,3.77,13,13
165,Alpine,4.00,13,13
167,Audi,4.31,13,13
174,Red Bull Racing,5.92,13,12
173,Racing Bulls,6.08,13,13
175,Williams,6.23,13,13
166,Aston Martin,6.69,13,13


### Slow corners found per session
The data picks the slow corners (median minimum speed below 120 km/h). Monza (Round 13) is a high-speed track, so few slow corners are expected there.

In [9]:
dr_q1 = corner_deficit_rows(corners, 'Q1')
dr_q1.drop_duplicates(['Round', 'Session'])[['Round', 'EventName', 'Session', 'SlowCorners']]

,Round,EventName,Session,SlowCorners
0,1,Australian Grand Prix,Q,3
11,2,Chinese Grand Prix,Q,7
22,2,Chinese Grand Prix,SQ,4
33,3,Japanese Grand Prix,Q,3
44,4,Miami Grand Prix,Q,9
55,4,Miami Grand Prix,SQ,9
66,5,Canadian Grand Prix,Q,6
77,5,Canadian Grand Prix,SQ,6
88,6,Monaco Grand Prix,Q,12
99,7,Barcelona Grand Prix,Q,3


### Decision: fastest lap from Q1 only vs the whole session
- **Q1**: every car runs it, all on the same tyre (Softs in Qualifying, Mediums in Sprint Qualifying), with similar track conditions.
- **all**: each team's best lap overall. Top teams set it later in the session on a faster, rubbered-in track, and in Sprint Qualifying the final round switches to Softs, so top teams get a tyre advantage.

Lower deficit is better.

In [10]:
cd = corner_deficit(corners, 'Q1').rename(columns={'score': 'Q1'}).join(corner_deficit(corners, 'all')['score'].rename('all'))
cd['Q1_rank'] = cd['Q1'].rank().astype(int)
cd['all_rank'] = cd['all'].rank().astype(int)
cd

,Q1,n,all,Q1_rank,all_rank
Team,,,,,
Ferrari,2.88,18,2.99,1,1
Mercedes,4.01,18,3.85,2,3
Audi,4.07,18,4.91,3,7
McLaren,4.18,18,3.70,4,2
Alpine,4.30,18,4.51,5,5
Haas F1 Team,4.82,18,5.68,6,8
Red Bull Racing,4.95,18,4.27,7,4
Racing Bulls,5.23,18,4.91,8,6
Aston Martin,5.76,18,7.00,9,9


### Consistency across sessions (Q1)
Is a team's deficit stable from track to track, or driven by one or two sessions?

In [11]:
dr_q1.pivot_table(index='Team', columns='Round', values='Deficit').round(1)

Round,1,2,3,4,5,6,7,9,10,11,12,13,15
Team,,,,,,,,,,,,,
Alpine,3.7,6.3,0.0,4.5,3.5,3.6,3.0,5.4,4.0,6.8,5.2,3.0,3.7
Aston Martin,6.7,4.0,8.0,6.0,6.4,5.4,5.3,5.3,7.2,7.7,5.6,2.5,6.2
Audi,4.3,3.8,6.0,3.8,9.7,3.4,3.3,1.4,1.8,4.5,3.2,2.0,3.8
Cadillac,8.0,6.2,6.0,7.0,7.6,3.6,5.0,6.9,3.8,5.8,5.0,4.0,6.6
Ferrari,0.5,1.8,1.0,3.3,6.3,4.1,2.7,1.8,3.8,1.5,2.3,2.0,5.3
Haas F1 Team,4.0,3.0,4.3,3.2,7.9,5.3,1.0,4.3,5.1,6.3,7.2,4.5,4.9
McLaren,2.3,4.0,NaN,4.5,6.5,2.6,3.7,5.6,4.3,2.0,3.9,3.0,4.1
Mercedes,2.5,6.1,2.3,4.3,7.5,4.9,0.7,4.4,5.2,2.7,3.5,2.0,2.4
Racing Bulls,4.7,5.4,4.4,4.8,8.8,3.8,4.3,4.0,6.0,2.7,5.2,4.0,7.8
